# PJM Hourly Load Forecasting — Demo

End-to-end walkthrough using the bundled sample CSV: load the data, build features, fit each of the three models, and compare them via walk-forward backtesting.

In [ ]:
import pandas as pd

from pjm_load_forecast import (
    load_pjm_csv, add_calendar_features, add_lag_features,
    add_rolling_features, build_design_matrix,
    SeasonalNaive, LinearModel, GradientBoostingModel,
    walk_forward_backtest,
)

In [ ]:
df = load_pjm_csv('../data/sample.csv')
df.head()

In [ ]:
feats = add_calendar_features(df)
feats = add_lag_features(feats, lags=[1, 24, 168])
feats = add_rolling_features(feats, windows=[24, 168])
feats = feats.dropna()
feats.head()

## Backtest each model

In [ ]:
models = {
    'naive':  SeasonalNaive(lag=168),
    'linear': LinearModel(alpha=1.0),
    'gbm':    GradientBoostingModel(max_iter=200, learning_rate=0.05),
}
rows = []
for name, model in models.items():
    r = walk_forward_backtest(model, feats, target='MW', train_size=200, horizon=24, step=48)
    rows.append({'model': name, 'mae': r['mae'], 'rmse': r['rmse'], 'mape': r['mape'], 'n_windows': r['n_windows']})
pd.DataFrame(rows)